# Cross-tabulation analysis of the FNF student survey

This notebook presents each planned cross-tabulation as a separate, clearly labelled table. It follows the one-table-at-a-time structure of `03_descriptive_analysis_simple.ipynb`.

The analysis is descriptive and unweighted. It does not test statistical significance.


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.precision", 1)

data_path = Path("../1_data/2_clean/student_survey_cleaned.dta")
df = pd.read_stata(data_path)
analysis_df = df.copy()

print("Dataset shape:", analysis_df.shape)


Dataset shape: (836, 114)


In [2]:
# Create age groups for the requested comparisons.
analysis_df["age_group"] = pd.cut(
    analysis_df["age"],
    bins=[17, 20, 23, 26, float("inf")],
    labels=["18-20", "21-23", "24-26", "27+"],
    include_lowest=True,
)

# Harmonize residual refusal codes used in this notebook.
for variable in ["q3_2", "q3_10_11"]:
    minus99 = analysis_df[variable].astype("string").eq("-99")
    analysis_df.loc[minus99, variable] = "Prefer not to say"
    analysis_df[variable] = analysis_df[variable].cat.remove_unused_categories()

print("Age groups:")
display(analysis_df["age_group"].value_counts(sort=False).to_frame("Frequency"))


Age groups:


,Frequency
age_group,
18-20,218
21-23,413
24-26,147
27+,58


In [3]:
required_variables = [
    "migration_plan", "gender", "education", "family_abroad", "life_place",
    "Province", "age_group", "q2_2", "q2_3", "q2_4", "q2_5", "q2_6",
    "q2_7", "q2_8_1", "q2_8_2", "q2_8_3", "q2_8_4", "q2_8_5",
    "q2_8_6", "q3_2", "q3_10_11", "q3_10_11_001", "q3_10_12",
    "q3_10_14", "q4_1", "q5_4", "ethnicity",
]

missing_variables = [
    variable for variable in required_variables
    if variable not in analysis_df.columns
]

if missing_variables:
    raise KeyError(f"Variables missing from the dataset: {missing_variables}")

print("All required variables were found.")


All required variables were found.


In [4]:
def crosstab_table(
    data,
    row_variable,
    column_variable,
    row_label,
    column_label,
):
    """Return one readable table containing counts and row percentages."""

    table_data = data[[row_variable, column_variable]].dropna().copy()

    for variable in [row_variable, column_variable]:
        if isinstance(table_data[variable].dtype, pd.CategoricalDtype):
            table_data[variable] = table_data[variable].cat.remove_unused_categories()

    counts = pd.crosstab(
        table_data[row_variable],
        table_data[column_variable],
    )

    row_percentages = (
        pd.crosstab(
            table_data[row_variable],
            table_data[column_variable],
            normalize="index",
        )
        .mul(100)
    )

    counts, row_percentages = counts.align(
        row_percentages,
        join="outer",
        fill_value=0,
    )

    percent_text = row_percentages.apply(
        lambda column: column.map(lambda value: f"{value:.1f}%")
    )

    formatted = (
        counts.astype(int).astype(str)
        + " ("
        + percent_text
        + ")"
    )

    formatted["Total n"] = counts.sum(axis=1).astype(int)
    formatted.index.name = row_label
    formatted.columns.name = column_label

    excluded = len(data) - len(table_data)
    print(f"Analytic N = {len(table_data):,}; excluded or routed = {excluded:,}")

    return formatted


## 1. Migration intention

Migration intention is shown across five demographic and migration-background characteristics.


### 1.1 Migration intention by gender


In [5]:
crosstab_table(
    data=analysis_df,
    row_variable="gender",
    column_variable="migration_plan",
    row_label="Gender",
    column_label="Migration intention",
)


Analytic N = 836; excluded or routed = 0


Migration intention,Go Abroad,Stay in Nepal,Don't Know,Total n
Gender,,,,
Male,181 (40.1%),210 (46.6%),60 (13.3%),451
Female,128 (33.2%),211 (54.8%),46 (11.9%),385


### 1.2 Migration intention by education level and year


In [6]:
crosstab_table(
    data=analysis_df,
    row_variable="education",
    column_variable="migration_plan",
    row_label="Education level and year",
    column_label="Migration intention",
)


Analytic N = 836; excluded or routed = 0


Migration intention,Go Abroad,Stay in Nepal,Don't Know,Total n
Education level and year,,,,
Bachelors 1st year,62 (40.0%),72 (46.5%),21 (13.5%),155
Bachelors 2nd year,65 (36.5%),78 (43.8%),35 (19.7%),178
Bachelors 3rd year,79 (37.3%),111 (52.4%),22 (10.4%),212
Bachelors 4th year,75 (41.2%),87 (47.8%),20 (11.0%),182
Bachelors 5th year,6 (37.5%),7 (43.8%),3 (18.8%),16
Masters 1st year,14 (22.2%),46 (73.0%),3 (4.8%),63
Masters 2nd year,8 (27.6%),19 (65.5%),2 (6.9%),29
PHD,0 (0.0%),1 (100.0%),0 (0.0%),1


### 1.3 Migration intention by family migration experience


In [7]:
crosstab_table(
    data=analysis_df,
    row_variable="family_abroad",
    column_variable="migration_plan",
    row_label="Close family member abroad",
    column_label="Migration intention",
)


Analytic N = 836; excluded or routed = 0


Migration intention,Go Abroad,Stay in Nepal,Don't Know,Total n
Close family member abroad,,,,
Yes,169 (47.6%),141 (39.7%),45 (12.7%),355
No,140 (29.1%),280 (58.2%),61 (12.7%),481


### 1.4 Migration intention by urban/rural background


In [8]:
crosstab_table(
    data=analysis_df,
    row_variable="life_place",
    column_variable="migration_plan",
    row_label="Urban/rural background",
    column_label="Migration intention",
)


Analytic N = 836; excluded or routed = 0


Migration intention,Go Abroad,Stay in Nepal,Don't Know,Total n
Urban/rural background,,,,
Urban,216 (44.4%),200 (41.1%),71 (14.6%),487
Rural,93 (26.6%),221 (63.3%),35 (10.0%),349


### 1.5 Migration intention by province


In [9]:
crosstab_table(
    data=analysis_df,
    row_variable="Province",
    column_variable="migration_plan",
    row_label="Province",
    column_label="Migration intention",
)


Analytic N = 834; excluded or routed = 2


Migration intention,Go Abroad,Stay in Nepal,Don't Know,Total n
Province,,,,
bagamati,119 (58.0%),55 (26.8%),31 (15.1%),205
gandaki,22 (34.9%),30 (47.6%),11 (17.5%),63
karnali,4 (8.9%),31 (68.9%),10 (22.2%),45
koshi,52 (38.2%),70 (51.5%),14 (10.3%),136
lumbini,45 (32.4%),81 (58.3%),13 (9.4%),139
madhesh,40 (24.0%),109 (65.3%),18 (10.8%),167
sudurpaschim,26 (32.9%),44 (55.7%),9 (11.4%),79


## 2. Political news consumption

The columns show frequency of following political news within each comparison group.


### 2.1 Political news consumption by migration intention


In [10]:
crosstab_table(
    data=analysis_df,
    row_variable="migration_plan",
    column_variable="q2_2",
    row_label="Migration intention",
    column_label="Political news consumption",
)


Analytic N = 836; excluded or routed = 0


Political news consumption,Daily,Several times a week,Weekly,Occasionally,Never,Total n
Migration intention,,,,,,
Go Abroad,151 (48.9%),81 (26.2%),12 (3.9%),58 (18.8%),7 (2.3%),309
Stay in Nepal,244 (58.0%),100 (23.8%),17 (4.0%),55 (13.1%),5 (1.2%),421
Don't Know,51 (48.1%),30 (28.3%),1 (0.9%),23 (21.7%),1 (0.9%),106


### 2.2 Political news consumption by gender


In [11]:
crosstab_table(
    data=analysis_df,
    row_variable="gender",
    column_variable="q2_2",
    row_label="Gender",
    column_label="Political news consumption",
)


Analytic N = 836; excluded or routed = 0


Political news consumption,Daily,Several times a week,Weekly,Occasionally,Never,Total n
Gender,,,,,,
Male,284 (63.0%),100 (22.2%),16 (3.5%),46 (10.2%),5 (1.1%),451
Female,162 (42.1%),111 (28.8%),14 (3.6%),90 (23.4%),8 (2.1%),385


### 2.3 Political news consumption by age group


In [12]:
crosstab_table(
    data=analysis_df,
    row_variable="age_group",
    column_variable="q2_2",
    row_label="Age group",
    column_label="Political news consumption",
)


Analytic N = 836; excluded or routed = 0


Political news consumption,Daily,Several times a week,Weekly,Occasionally,Never,Total n
Age group,,,,,,
18-20,90 (41.3%),53 (24.3%),15 (6.9%),53 (24.3%),7 (3.2%),218
21-23,212 (51.3%),128 (31.0%),8 (1.9%),61 (14.8%),4 (1.0%),413
24-26,94 (63.9%),23 (15.6%),7 (4.8%),21 (14.3%),2 (1.4%),147
27+,50 (86.2%),7 (12.1%),0 (0.0%),1 (1.7%),0 (0.0%),58


### 2.4 Political news consumption by education level and year


In [13]:
crosstab_table(
    data=analysis_df,
    row_variable="education",
    column_variable="q2_2",
    row_label="Education level and year",
    column_label="Political news consumption",
)


Analytic N = 836; excluded or routed = 0


Political news consumption,Daily,Several times a week,Weekly,Occasionally,Never,Total n
Education level and year,,,,,,
Bachelors 1st year,61 (39.4%),39 (25.2%),12 (7.7%),38 (24.5%),5 (3.2%),155
Bachelors 2nd year,77 (43.3%),55 (30.9%),6 (3.4%),36 (20.2%),4 (2.2%),178
Bachelors 3rd year,114 (53.8%),58 (27.4%),7 (3.3%),31 (14.6%),2 (0.9%),212
Bachelors 4th year,116 (63.7%),42 (23.1%),0 (0.0%),22 (12.1%),2 (1.1%),182
Bachelors 5th year,7 (43.8%),3 (18.8%),1 (6.2%),5 (31.2%),0 (0.0%),16
Masters 1st year,43 (68.3%),12 (19.0%),4 (6.3%),4 (6.3%),0 (0.0%),63
Masters 2nd year,27 (93.1%),2 (6.9%),0 (0.0%),0 (0.0%),0 (0.0%),29
PHD,1 (100.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1


### 2.5 Political news consumption by province


In [14]:
crosstab_table(
    data=analysis_df,
    row_variable="Province",
    column_variable="q2_2",
    row_label="Province",
    column_label="Political news consumption",
)


Analytic N = 834; excluded or routed = 2


Political news consumption,Daily,Several times a week,Weekly,Occasionally,Never,Total n
Province,,,,,,
bagamati,74 (36.1%),58 (28.3%),6 (2.9%),61 (29.8%),6 (2.9%),205
gandaki,30 (47.6%),25 (39.7%),1 (1.6%),7 (11.1%),0 (0.0%),63
karnali,21 (46.7%),5 (11.1%),0 (0.0%),16 (35.6%),3 (6.7%),45
koshi,96 (70.6%),19 (14.0%),5 (3.7%),15 (11.0%),1 (0.7%),136
lumbini,97 (69.8%),36 (25.9%),4 (2.9%),2 (1.4%),0 (0.0%),139
madhesh,103 (61.7%),36 (21.6%),9 (5.4%),18 (10.8%),1 (0.6%),167
sudurpaschim,24 (30.4%),32 (40.5%),4 (5.1%),17 (21.5%),2 (2.5%),79


## 3. Political or organizational involvement

The columns show whether respondents reported active political or organizational involvement.


### 3.1 Political involvement by migration intention


In [15]:
crosstab_table(
    data=analysis_df,
    row_variable="migration_plan",
    column_variable="q2_3",
    row_label="Migration intention",
    column_label="Political involvement",
)


Analytic N = 836; excluded or routed = 0


Political involvement,Yes,No,Total n
Migration intention,,,
Go Abroad,73 (23.6%),236 (76.4%),309
Stay in Nepal,81 (19.2%),340 (80.8%),421
Don't Know,15 (14.2%),91 (85.8%),106


### 3.2 Political involvement by gender


In [16]:
crosstab_table(
    data=analysis_df,
    row_variable="gender",
    column_variable="q2_3",
    row_label="Gender",
    column_label="Political involvement",
)


Analytic N = 836; excluded or routed = 0


Political involvement,Yes,No,Total n
Gender,,,
Male,126 (27.9%),325 (72.1%),451
Female,43 (11.2%),342 (88.8%),385


### 3.3 Political involvement by education level and year


In [17]:
crosstab_table(
    data=analysis_df,
    row_variable="education",
    column_variable="q2_3",
    row_label="Education level and year",
    column_label="Political involvement",
)


Analytic N = 836; excluded or routed = 0


Political involvement,Yes,No,Total n
Education level and year,,,
Bachelors 1st year,17 (11.0%),138 (89.0%),155
Bachelors 2nd year,20 (11.2%),158 (88.8%),178
Bachelors 3rd year,30 (14.2%),182 (85.8%),212
Bachelors 4th year,49 (26.9%),133 (73.1%),182
Bachelors 5th year,6 (37.5%),10 (62.5%),16
Masters 1st year,28 (44.4%),35 (55.6%),63
Masters 2nd year,18 (62.1%),11 (37.9%),29
PHD,1 (100.0%),0 (0.0%),1


### 3.4 Political involvement by urban/rural background


In [18]:
crosstab_table(
    data=analysis_df,
    row_variable="life_place",
    column_variable="q2_3",
    row_label="Urban/rural background",
    column_label="Political involvement",
)


Analytic N = 836; excluded or routed = 0


Political involvement,Yes,No,Total n
Urban/rural background,,,
Urban,105 (21.6%),382 (78.4%),487
Rural,64 (18.3%),285 (81.7%),349


## 4. Perceptions of political stability in Nepal today

The columns show assessments of Nepal's current political stability. Migration intention appeared twice in the original list and is included once here.


### 4.1 Political stability today by migration intention


In [19]:
crosstab_table(
    data=analysis_df,
    row_variable="migration_plan",
    column_variable="q3_2",
    row_label="Migration intention",
    column_label="Political stability today",
)


Analytic N = 836; excluded or routed = 0


Political stability today,Very Stable,Stable,Neutral,Unstable,Very Unstable,Prefer not to say,Total n
Migration intention,,,,,,,
Go Abroad,10 (3.2%),166 (53.7%),76 (24.6%),45 (14.6%),10 (3.2%),2 (0.6%),309
Stay in Nepal,24 (5.7%),239 (56.8%),75 (17.8%),64 (15.2%),13 (3.1%),6 (1.4%),421
Don't Know,9 (8.5%),49 (46.2%),33 (31.1%),10 (9.4%),3 (2.8%),2 (1.9%),106


### 4.2 Political stability today by political news consumption


In [20]:
crosstab_table(
    data=analysis_df,
    row_variable="q2_2",
    column_variable="q3_2",
    row_label="Political news consumption",
    column_label="Political stability today",
)


Analytic N = 836; excluded or routed = 0


Political stability today,Very Stable,Stable,Neutral,Unstable,Very Unstable,Prefer not to say,Total n
Political news consumption,,,,,,,
Daily,27 (6.1%),257 (57.6%),88 (19.7%),53 (11.9%),20 (4.5%),1 (0.2%),446
Several times a week,4 (1.9%),123 (58.3%),43 (20.4%),38 (18.0%),1 (0.5%),2 (0.9%),211
Weekly,2 (6.7%),16 (53.3%),4 (13.3%),8 (26.7%),0 (0.0%),0 (0.0%),30
Occasionally,10 (7.4%),55 (40.4%),40 (29.4%),20 (14.7%),4 (2.9%),7 (5.1%),136
Never,0 (0.0%),3 (23.1%),9 (69.2%),0 (0.0%),1 (7.7%),0 (0.0%),13


### 4.3 Political stability today by gender


In [21]:
crosstab_table(
    data=analysis_df,
    row_variable="gender",
    column_variable="q3_2",
    row_label="Gender",
    column_label="Political stability today",
)


Analytic N = 836; excluded or routed = 0


Political stability today,Very Stable,Stable,Neutral,Unstable,Very Unstable,Prefer not to say,Total n
Gender,,,,,,,
Male,26 (5.8%),235 (52.1%),96 (21.3%),70 (15.5%),20 (4.4%),4 (0.9%),451
Female,17 (4.4%),219 (56.9%),88 (22.9%),49 (12.7%),6 (1.6%),6 (1.6%),385


### 4.4 Political stability today by province


In [22]:
crosstab_table(
    data=analysis_df,
    row_variable="Province",
    column_variable="q3_2",
    row_label="Province",
    column_label="Political stability today",
)


Analytic N = 834; excluded or routed = 2


Political stability today,Very Stable,Stable,Neutral,Unstable,Very Unstable,Prefer not to say,Total n
Province,,,,,,,
bagamati,2 (1.0%),79 (38.5%),66 (32.2%),44 (21.5%),10 (4.9%),4 (2.0%),205
gandaki,2 (3.2%),42 (66.7%),14 (22.2%),5 (7.9%),0 (0.0%),0 (0.0%),63
karnali,2 (4.4%),23 (51.1%),11 (24.4%),6 (13.3%),3 (6.7%),0 (0.0%),45
koshi,16 (11.8%),55 (40.4%),33 (24.3%),22 (16.2%),10 (7.4%),0 (0.0%),136
lumbini,1 (0.7%),100 (71.9%),30 (21.6%),8 (5.8%),0 (0.0%),0 (0.0%),139
madhesh,14 (8.4%),115 (68.9%),21 (12.6%),9 (5.4%),2 (1.2%),6 (3.6%),167
sudurpaschim,5 (6.3%),39 (49.4%),9 (11.4%),25 (31.6%),1 (1.3%),0 (0.0%),79


### 4.5 Political stability today by education level and year


In [23]:
crosstab_table(
    data=analysis_df,
    row_variable="education",
    column_variable="q3_2",
    row_label="Education level and year",
    column_label="Political stability today",
)


Analytic N = 836; excluded or routed = 0


Political stability today,Very Stable,Stable,Neutral,Unstable,Very Unstable,Prefer not to say,Total n
Education level and year,,,,,,,
Bachelors 1st year,15 (9.7%),83 (53.5%),30 (19.4%),20 (12.9%),5 (3.2%),2 (1.3%),155
Bachelors 2nd year,7 (3.9%),89 (50.0%),54 (30.3%),24 (13.5%),3 (1.7%),1 (0.6%),178
Bachelors 3rd year,8 (3.8%),133 (62.7%),39 (18.4%),24 (11.3%),3 (1.4%),5 (2.4%),212
Bachelors 4th year,10 (5.5%),99 (54.4%),44 (24.2%),21 (11.5%),6 (3.3%),2 (1.1%),182
Bachelors 5th year,0 (0.0%),8 (50.0%),4 (25.0%),3 (18.8%),1 (6.2%),0 (0.0%),16
Masters 1st year,3 (4.8%),32 (50.8%),7 (11.1%),18 (28.6%),3 (4.8%),0 (0.0%),63
Masters 2nd year,0 (0.0%),10 (34.5%),6 (20.7%),8 (27.6%),5 (17.2%),0 (0.0%),29
PHD,0 (0.0%),0 (0.0%),0 (0.0%),1 (100.0%),0 (0.0%),0 (0.0%),1


### 4.6 Political stability today by ethnicity


In [24]:
crosstab_table(
    data=analysis_df,
    row_variable="ethnicity",
    column_variable="q3_2",
    row_label="Ethnicity",
    column_label="Political stability today",
)


Analytic N = 835; excluded or routed = 1


Political stability today,Very Stable,Stable,Neutral,Unstable,Very Unstable,Prefer not to say,Total n
Ethnicity,,,,,,,
Bhramin/Chettri,12 (3.4%),187 (52.8%),87 (24.6%),50 (14.1%),14 (4.0%),4 (1.1%),354
Dalit,4 (6.5%),30 (48.4%),15 (24.2%),12 (19.4%),1 (1.6%),0 (0.0%),62
Janajati,2 (1.3%),74 (48.1%),43 (27.9%),30 (19.5%),5 (3.2%),0 (0.0%),154
Muslim,0 (0.0%),12 (66.7%),2 (11.1%),3 (16.7%),1 (5.6%),0 (0.0%),18
Madhesi,25 (10.1%),150 (60.7%),37 (15.0%),24 (9.7%),5 (2.0%),6 (2.4%),247


## 5. Attention to the Gen-Z protest

The columns show how closely respondents followed or engaged with the September 8 Gen-Z protest and subsequent politics.


### 5.1 Gen-Z protest attention by political stability perception


In [25]:
crosstab_table(
    data=analysis_df,
    row_variable="q3_2",
    column_variable="q4_1",
    row_label="Political stability today",
    column_label="Gen-Z protest attention",
)


Analytic N = 836; excluded or routed = 0


Gen-Z protest attention,Engaged Directly,Very Closely,Somewhat Closely,Heard But followed Vary Little,Not at All,Total n
Political stability today,,,,,,
Very Stable,19 (44.2%),20 (46.5%),4 (9.3%),0 (0.0%),0 (0.0%),43
Stable,120 (26.4%),271 (59.7%),52 (11.5%),8 (1.8%),3 (0.7%),454
Neutral,46 (25.0%),98 (53.3%),34 (18.5%),6 (3.3%),0 (0.0%),184
Unstable,27 (22.7%),69 (58.0%),21 (17.6%),1 (0.8%),1 (0.8%),119
Very Unstable,12 (46.2%),12 (46.2%),2 (7.7%),0 (0.0%),0 (0.0%),26
Prefer not to say,0 (0.0%),7 (70.0%),3 (30.0%),0 (0.0%),0 (0.0%),10


### 5.2 Gen-Z protest attention by migration intention


In [26]:
crosstab_table(
    data=analysis_df,
    row_variable="migration_plan",
    column_variable="q4_1",
    row_label="Migration intention",
    column_label="Gen-Z protest attention",
)


Analytic N = 836; excluded or routed = 0


Gen-Z protest attention,Engaged Directly,Very Closely,Somewhat Closely,Heard But followed Vary Little,Not at All,Total n
Migration intention,,,,,,
Go Abroad,115 (37.2%),152 (49.2%),37 (12.0%),4 (1.3%),1 (0.3%),309
Stay in Nepal,86 (20.4%),259 (61.5%),63 (15.0%),10 (2.4%),3 (0.7%),421
Don't Know,23 (21.7%),66 (62.3%),16 (15.1%),1 (0.9%),0 (0.0%),106


### 5.3 Gen-Z protest attention by gender


In [27]:
crosstab_table(
    data=analysis_df,
    row_variable="gender",
    column_variable="q4_1",
    row_label="Gender",
    column_label="Gen-Z protest attention",
)


Analytic N = 836; excluded or routed = 0


Gen-Z protest attention,Engaged Directly,Very Closely,Somewhat Closely,Heard But followed Vary Little,Not at All,Total n
Gender,,,,,,
Male,155 (34.4%),255 (56.5%),39 (8.6%),2 (0.4%),0 (0.0%),451
Female,69 (17.9%),222 (57.7%),77 (20.0%),13 (3.4%),4 (1.0%),385


### 5.4 Gen-Z protest attention by age group


In [28]:
crosstab_table(
    data=analysis_df,
    row_variable="age_group",
    column_variable="q4_1",
    row_label="Age group",
    column_label="Gen-Z protest attention",
)


Analytic N = 836; excluded or routed = 0


Gen-Z protest attention,Engaged Directly,Very Closely,Somewhat Closely,Heard But followed Vary Little,Not at All,Total n
Age group,,,,,,
18-20,47 (21.6%),119 (54.6%),47 (21.6%),3 (1.4%),2 (0.9%),218
21-23,101 (24.5%),252 (61.0%),49 (11.9%),9 (2.2%),2 (0.5%),413
24-26,57 (38.8%),71 (48.3%),16 (10.9%),3 (2.0%),0 (0.0%),147
27+,19 (32.8%),35 (60.3%),4 (6.9%),0 (0.0%),0 (0.0%),58


### 5.5 Gen-Z protest attention by education level and year


In [29]:
crosstab_table(
    data=analysis_df,
    row_variable="education",
    column_variable="q4_1",
    row_label="Education level and year",
    column_label="Gen-Z protest attention",
)


Analytic N = 836; excluded or routed = 0


Gen-Z protest attention,Engaged Directly,Very Closely,Somewhat Closely,Heard But followed Vary Little,Not at All,Total n
Education level and year,,,,,,
Bachelors 1st year,34 (21.9%),88 (56.8%),31 (20.0%),2 (1.3%),0 (0.0%),155
Bachelors 2nd year,44 (24.7%),98 (55.1%),32 (18.0%),4 (2.2%),0 (0.0%),178
Bachelors 3rd year,51 (24.1%),133 (62.7%),20 (9.4%),6 (2.8%),2 (0.9%),212
Bachelors 4th year,60 (33.0%),95 (52.2%),22 (12.1%),3 (1.6%),2 (1.1%),182
Bachelors 5th year,7 (43.8%),8 (50.0%),1 (6.2%),0 (0.0%),0 (0.0%),16
Masters 1st year,20 (31.7%),34 (54.0%),9 (14.3%),0 (0.0%),0 (0.0%),63
Masters 2nd year,8 (27.6%),20 (69.0%),1 (3.4%),0 (0.0%),0 (0.0%),29
PHD,0 (0.0%),1 (100.0%),0 (0.0%),0 (0.0%),0 (0.0%),1


### 5.6 Gen-Z protest attention by political involvement


In [30]:
crosstab_table(
    data=analysis_df,
    row_variable="q2_3",
    column_variable="q4_1",
    row_label="Political involvement",
    column_label="Gen-Z protest attention",
)


Analytic N = 836; excluded or routed = 0


Gen-Z protest attention,Engaged Directly,Very Closely,Somewhat Closely,Heard But followed Vary Little,Not at All,Total n
Political involvement,,,,,,
Yes,77 (45.6%),86 (50.9%),3 (1.8%),1 (0.6%),2 (1.2%),169
No,147 (22.0%),391 (58.6%),113 (16.9%),14 (2.1%),2 (0.3%),667


## 6. Past voting and future voting intention

Election questions have different routed denominators. Federal and local voting tables use 460 valid respondents; the provincial item was asked only among 228 respondents who reported voting federally.


### 6.1 Federal voting in 2022 by future local-election intention


In [31]:
crosstab_table(
    data=analysis_df,
    row_variable="q2_4",
    column_variable="q2_7",
    row_label="Voted in 2022 federal election",
    column_label="Future voting intention",
)


Analytic N = 460; excluded or routed = 376


Future voting intention,Very Likely,Somewhat Likely,Somewhat Unlikely,Very Unlikely,Donot Know,Total n
Voted in 2022 federal election,,,,,,
Yes,188 (82.5%),33 (14.5%),1 (0.4%),3 (1.3%),3 (1.3%),228
No,124 (55.4%),49 (21.9%),14 (6.2%),21 (9.4%),16 (7.1%),224
Not Sure,6 (75.0%),2 (25.0%),0 (0.0%),0 (0.0%),0 (0.0%),8


### 6.2 Provincial voting in 2022 by future local-election intention


In [32]:
crosstab_table(
    data=analysis_df,
    row_variable="q2_5",
    column_variable="q2_7",
    row_label="Voted in 2022 provincial election",
    column_label="Future voting intention",
)


Analytic N = 228; excluded or routed = 608


Future voting intention,Very Likely,Somewhat Likely,Somewhat Unlikely,Very Unlikely,Donot Know,Total n
Voted in 2022 provincial election,,,,,,
Yes,182 (83.9%),28 (12.9%),1 (0.5%),3 (1.4%),3 (1.4%),217
No,5 (50.0%),5 (50.0%),0 (0.0%),0 (0.0%),0 (0.0%),10
Not Sure,1 (100.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1


### 6.3 Local voting in 2022 by future local-election intention


In [33]:
crosstab_table(
    data=analysis_df,
    row_variable="q2_6",
    column_variable="q2_7",
    row_label="Voted in 2022 local election",
    column_label="Future voting intention",
)


Analytic N = 460; excluded or routed = 376


Future voting intention,Very Likely,Somewhat Likely,Somewhat Unlikely,Very Unlikely,Donot Know,Total n
Voted in 2022 local election,,,,,,
Yes,184 (82.9%),31 (14.0%),1 (0.5%),3 (1.4%),3 (1.4%),222
No,127 (55.7%),51 (22.4%),14 (6.1%),21 (9.2%),15 (6.6%),228
Not Sure,7 (70.0%),2 (20.0%),0 (0.0%),0 (0.0%),1 (10.0%),10


## 7. Political news consumption and civic participation

Each table shows Yes/No participation within each political-news-consumption group.


### 7.1 Political news and attendance at a campaign or rally


In [34]:
crosstab_table(
    data=analysis_df,
    row_variable="q2_2",
    column_variable="q2_8_1",
    row_label="Political news consumption",
    column_label="Attended campaign/rally",
)


Analytic N = 836; excluded or routed = 0


Attended campaign/rally,Yes,No,Total n
Political news consumption,,,
Daily,135 (30.3%),311 (69.7%),446
Several times a week,28 (13.3%),183 (86.7%),211
Weekly,3 (10.0%),27 (90.0%),30
Occasionally,9 (6.6%),127 (93.4%),136
Never,0 (0.0%),13 (100.0%),13


### 7.2 Political news and volunteering for a party or campaign


In [35]:
crosstab_table(
    data=analysis_df,
    row_variable="q2_2",
    column_variable="q2_8_2",
    row_label="Political news consumption",
    column_label="Volunteered for party/campaign",
)


Analytic N = 836; excluded or routed = 0


Volunteered for party/campaign,Yes,No,Total n
Political news consumption,,,
Daily,77 (17.3%),369 (82.7%),446
Several times a week,7 (3.3%),204 (96.7%),211
Weekly,2 (6.7%),28 (93.3%),30
Occasionally,3 (2.2%),133 (97.8%),136
Never,0 (0.0%),13 (100.0%),13


### 7.3 Political news and joining a protest or demonstration


In [36]:
crosstab_table(
    data=analysis_df,
    row_variable="q2_2",
    column_variable="q2_8_3",
    row_label="Political news consumption",
    column_label="Joined protest/demonstration",
)


Analytic N = 836; excluded or routed = 0


Joined protest/demonstration,Yes,No,Total n
Political news consumption,,,
Daily,166 (37.2%),280 (62.8%),446
Several times a week,54 (25.6%),157 (74.4%),211
Weekly,5 (16.7%),25 (83.3%),30
Occasionally,19 (14.0%),117 (86.0%),136
Never,2 (15.4%),11 (84.6%),13


### 7.4 Political news and joining a public meeting or discussion


In [37]:
crosstab_table(
    data=analysis_df,
    row_variable="q2_2",
    column_variable="q2_8_4",
    row_label="Political news consumption",
    column_label="Joined public meeting/discussion",
)


Analytic N = 836; excluded or routed = 0


Joined public meeting/discussion,Yes,No,Total n
Political news consumption,,,
Daily,185 (41.5%),261 (58.5%),446
Several times a week,35 (16.6%),176 (83.4%),211
Weekly,5 (16.7%),25 (83.3%),30
Occasionally,39 (28.7%),97 (71.3%),136
Never,1 (7.7%),12 (92.3%),13


### 7.5 Political news and contacting an elected representative


In [38]:
crosstab_table(
    data=analysis_df,
    row_variable="q2_2",
    column_variable="q2_8_5",
    row_label="Political news consumption",
    column_label="Contacted elected representative",
)


Analytic N = 836; excluded or routed = 0


Contacted elected representative,Yes,No,Total n
Political news consumption,,,
Daily,121 (27.1%),325 (72.9%),446
Several times a week,26 (12.3%),185 (87.7%),211
Weekly,4 (13.3%),26 (86.7%),30
Occasionally,17 (12.5%),119 (87.5%),136
Never,1 (7.7%),12 (92.3%),13


### 7.6 Political news and joining a ward user committee


In [39]:
crosstab_table(
    data=analysis_df,
    row_variable="q2_2",
    column_variable="q2_8_6",
    row_label="Political news consumption",
    column_label="Joined ward user committee",
)


Analytic N = 836; excluded or routed = 0


Joined ward user committee,Yes,No,Total n
Political news consumption,,,
Daily,32 (7.2%),414 (92.8%),446
Several times a week,3 (1.4%),208 (98.6%),211
Weekly,0 (0.0%),30 (100.0%),30
Occasionally,2 (1.5%),134 (98.5%),136
Never,0 (0.0%),13 (100.0%),13


## 8. Nepal and South Asian stability perceptions

Caution: q5_4 is labelled as South Asian stability today, but its recorded response categories are comparative (for example, 'Much more stable'). Confirm the intended reference point in the questionnaire before interpreting this table.


### 8.1 Nepal stability today by South Asian stability perception


In [40]:
crosstab_table(
    data=analysis_df,
    row_variable="q3_2",
    column_variable="q5_4",
    row_label="Nepal political stability today",
    column_label="South Asian stability perception",
)


Analytic N = 836; excluded or routed = 0


South Asian stability perception,Much more stable,Somewhat more stable,About the same,Somewhat less stable,Much less stable,Prefer not to say,Total n
Nepal political stability today,,,,,,,
Very Stable,7 (16.3%),23 (53.5%),5 (11.6%),8 (18.6%),0 (0.0%),0 (0.0%),43
Stable,11 (2.4%),254 (55.9%),88 (19.4%),51 (11.2%),5 (1.1%),45 (9.9%),454
Neutral,4 (2.2%),57 (31.0%),65 (35.3%),45 (24.5%),3 (1.6%),10 (5.4%),184
Unstable,1 (0.8%),30 (25.2%),22 (18.5%),54 (45.4%),4 (3.4%),8 (6.7%),119
Very Unstable,0 (0.0%),8 (30.8%),3 (11.5%),8 (30.8%),6 (23.1%),1 (3.8%),26
Prefer not to say,0 (0.0%),4 (40.0%),1 (10.0%),2 (20.0%),0 (0.0%),3 (30.0%),10


## 9. Migration intention and perceived risks to Nepal's stability

The concern items are Likert statements. Governance is represented separately by corruption and lack of accountability.


### 9.1 Migration intention by concern about corruption


In [41]:
crosstab_table(
    data=analysis_df,
    row_variable="migration_plan",
    column_variable="q3_10_11",
    row_label="Migration intention",
    column_label="Corruption poses a stability risk",
)


Analytic N = 836; excluded or routed = 0


Corruption poses a stability risk,Strongly Agree,Agree,Neutral,Disagree,Strongly Disagree,Prefer not to say,Total n
Migration intention,,,,,,,
Go Abroad,214 (69.3%),87 (28.2%),5 (1.6%),3 (1.0%),0 (0.0%),0 (0.0%),309
Stay in Nepal,254 (60.3%),148 (35.2%),9 (2.1%),6 (1.4%),2 (0.5%),2 (0.5%),421
Don't Know,67 (63.2%),31 (29.2%),4 (3.8%),3 (2.8%),1 (0.9%),0 (0.0%),106


### 9.2 Migration intention by concern about lack of accountability


In [42]:
crosstab_table(
    data=analysis_df,
    row_variable="migration_plan",
    column_variable="q3_10_11_001",
    row_label="Migration intention",
    column_label="Lack of accountability poses a stability risk",
)


Analytic N = 836; excluded or routed = 0


Lack of accountability poses a stability risk,Strongly Agree,Agree,Neutral,Disagree,Strongly Disagree,Donot Know,Total n
Migration intention,,,,,,,
Go Abroad,145 (46.9%),152 (49.2%),9 (2.9%),3 (1.0%),0 (0.0%),0 (0.0%),309
Stay in Nepal,204 (48.5%),190 (45.1%),19 (4.5%),5 (1.2%),2 (0.5%),1 (0.2%),421
Don't Know,45 (42.5%),50 (47.2%),8 (7.5%),2 (1.9%),1 (0.9%),0 (0.0%),106


### 9.3 Migration intention by concern about unemployment and economic opportunities


In [43]:
crosstab_table(
    data=analysis_df,
    row_variable="migration_plan",
    column_variable="q3_10_12",
    row_label="Migration intention",
    column_label="Unemployment/economic opportunities pose a stability risk",
)


Analytic N = 836; excluded or routed = 0


Unemployment/economic opportunities pose a stability risk,Strongly Agree,Agree,Neutral,Disagree,Strongly Disagree,Prefer not to say,Total n
Migration intention,,,,,,,
Go Abroad,167 (54.0%),123 (39.8%),10 (3.2%),9 (2.9%),0 (0.0%),0 (0.0%),309
Stay in Nepal,218 (51.8%),182 (43.2%),15 (3.6%),4 (1.0%),2 (0.5%),0 (0.0%),421
Don't Know,60 (56.6%),40 (37.7%),2 (1.9%),2 (1.9%),1 (0.9%),1 (0.9%),106


### 9.4 Migration intention by concern about rising cost of living


In [44]:
crosstab_table(
    data=analysis_df,
    row_variable="migration_plan",
    column_variable="q3_10_14",
    row_label="Migration intention",
    column_label="Rising cost of living poses a stability risk",
)


Analytic N = 836; excluded or routed = 0


Rising cost of living poses a stability risk,Strongly Agree,Agree,Neutral,Disagree,Strongly Disagree,Prefer not to say,Total n
Migration intention,,,,,,,
Go Abroad,133 (43.0%),139 (45.0%),20 (6.5%),13 (4.2%),4 (1.3%),0 (0.0%),309
Stay in Nepal,155 (36.8%),222 (52.7%),31 (7.4%),11 (2.6%),1 (0.2%),1 (0.2%),421
Don't Know,42 (39.6%),45 (42.5%),12 (11.3%),6 (5.7%),1 (0.9%),0 (0.0%),106


## End of cross-tabulation review

These tables describe patterns within the achieved sample. They should not be interpreted as population estimates or evidence of statistically significant relationships. Chi-square tests and effect sizes can be added after reviewing cell sizes and finalizing the comparison plan.
